# Get airoports data
[Open airoports](https://ourairports.com/data/)



In [ ]:
import pandas as pd
df = pd.read_csv("./data/airports.csv")
df.head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN


In [2]:
df = df[['icao_code', 'name', 'type', 'iso_country', 'iso_region', 'municipality', 'latitude_deg', 'longitude_deg']]

from graph_creation.config import EU_COUNTRY_CODES
only_eu_mask = df['iso_country'].apply(lambda x: x in EU_COUNTRY_CODES)
only_aeroports_mask = df['type'].apply(lambda x: x in ['small_airport', 'medium_airport', 'large_airport'])
icao_code_exists_mask = ~df['icao_code'].isna()

mask = only_eu_mask & only_aeroports_mask & icao_code_exists_mask

eu_aeroports_df = df[mask]
eu_aeroports_df = eu_aeroports_df[~eu_aeroports_df['municipality'].isna()]
eu_aeroports_df.head()

,icao_code,name,type,iso_country,iso_region,municipality,latitude_deg,longitude_deg
12527,EBLB,Elsenborn Air Base,small_airport,BE,BE-WLG,Bütgenbach,50.484818,6.183014
12580,LBDM,Dve Mogili Airfield,small_airport,BG,BG-18,Dve Mogili,43.606541,25.890690
12592,LBBB,Belchinski Bani Airstrip,small_airport,BG,BG-23,Belchinski Bani,42.374713,23.392548
12594,LBBL,Blagoevo Airfield,small_airport,BG,BG-17,Blagoevo,43.455488,26.431655
12597,LBDR,Draganovtsi Airfield,small_airport,BG,BG-07,Draganovtsi,42.943042,25.169528


In [3]:
eu_aeroports_df.shape

(1325, 8)

In [4]:
eu_aeroports_df.isna().sum().sum()

np.int64(0)

In [5]:
eu_aeroports_df.to_csv("./data/eu_aeroports.csv", index=False)

## Aeroports to municipality aggregation

In [6]:
eu_aeroports_df['type'].value_counts()

type
small_airport     669
medium_airport    441
large_airport     215
Name: count, dtype: int64

In [7]:
# the same as eu_aeroports_df[['municipality']].value_counts().value_counts()
eu_aeroports_df.groupby('municipality')['icao_code'].count().value_counts()

icao_code
1    1259
2      25
3       4
4       1
Name: count, dtype: int64

In [8]:
eu_aeroports_df[['municipality', 'iso_country']].value_counts().value_counts()

count
1    1261
2      24
3       4
4       1
Name: count, dtype: int64

In [9]:
eu_aeroports_df[eu_aeroports_df['municipality'] == 'Mora']

,icao_code,name,type,iso_country,iso_region,municipality,latitude_deg,longitude_deg
25073,ESKM,Mora Airport,medium_airport,SE,SE-W,Mora,60.957901,14.51140
44086,LPMO,Morargil Airfield,small_airport,PT,PT-07,Mora,38.993532,-8.14201


In [17]:
municipality_df = (
    eu_aeroports_df
    .groupby(["municipality", "iso_country"], as_index=False)
    .agg(
        icao_codes=("icao_code", list),
        iso_country=("iso_country", "first"),
        iso_region=("iso_region", "first"),
        latitude_deg=("latitude_deg", "mean"),
        longitude_deg=("longitude_deg", "mean"),
    )
)
municipality_df.head()

,municipality,icao_codes,iso_country,iso_region,latitude_deg,longitude_deg
0,Aachen,[EDKA],DE,DE-NW,50.823055,6.186389
1,Aalborg,[EKYT],DK,DK-81,57.094763,9.849930
2,Aarhus,[EKAH],DK,DK-82,56.303331,10.618286
3,Abbeyshrule,[EIAB],IE,IE-LD,53.591373,-7.642947
4,Aberdeen,[EGPD],GB,GB-SCT,57.201900,-2.197780


In [19]:
municipality_df['icao_codes'] = municipality_df['icao_codes'].apply(lambda x: ", ".join(x))

In [20]:
municipality_df[municipality_df['municipality'] == 'Mora']

,municipality,icao_codes,iso_country,iso_region,latitude_deg,longitude_deg
758,Mora,LPMO,PT,PT-07,38.993532,-8.14201
759,Mora,ESKM,SE,SE-W,60.957901,14.51140


In [21]:
municipality_df.rename({"icao_codes": "airports"}, inplace=True, axis=1)

In [22]:
municipality_df.to_csv("./data/municipality.csv", index=False)

# Get LAU to NUT3 df

In [ ]:
import pandas as pd

input_path = "./data/eu_27_lau_nuts.xlsx"

# read all sheets
xls = pd.ExcelFile(input_path)

frames = []

for sheet in xls.sheet_names:
    if len(sheet) != 2:
        continue  # skip metadata sheets

    country_code = sheet[1:]  # "0DE" -> "DE"

    df = pd.read_excel(
        xls,
        sheet_name=sheet,
        dtype=str
    )

    # normalize column names (important!)
    df.columns = df.columns.str.strip().str.upper()

    # select & rename
    sub = df[[
        "NUTS3",
        "LAU NAME NATIONAL"
    ]].copy()

    sub["COUNTRY_CODE"] = country_code

    frames.append(sub)

result = pd.concat(frames, ignore_index=True)

# Graph creation

In [2]:
from graph_creation import bootsrap_create_save_graph_use_case
u = bootsrap_create_save_graph_use_case()

u.run("2026-02-24", graph_name="2026-24-02_graph")

sblenlkj-api-client LXjjqvHwoWCjIEmlXXZaGKMe1UjeRci0
loaded 99356 flights from OpenSky
flights with empty arr/dep = 29946
unknown or non-eu departures: 58393
unknown or non-eu arrivals: 3051


In [1]:
from graph_creation.adapters.input import GraphJsonLoader
g = GraphJsonLoader().load("./data/graph.json")
print(g)

Graph(nodes=520, edges=4588, unknown_or_non_eu_dep=0, unknown_or_non_eu_arr=0, begin=None, end=None)


In [2]:
for i, node in enumerate(g.nodes):
    print(f"Node {i}: {node}")
    if i == 5:
        break

Node 0: Node(id='Aachen, DE', name='Aachen', iso_country='DE', iso_region='DE-NW', latitude=50.823055, longitude=6.186389, airports=['EDKA'], weight=1)
Node 1: Node(id='Aalborg, DK', name='Aalborg', iso_country='DK', iso_region='DK-81', latitude=57.094763, longitude=9.84993, airports=['EKYT'], weight=24)
Node 2: Node(id='Aarhus, DK', name='Aarhus', iso_country='DK', iso_region='DK-82', latitude=56.303331, longitude=10.618286, airports=['EKAH'], weight=6)
Node 3: Node(id='Aberdeen, GB', name='Aberdeen', iso_country='GB', iso_region='GB-SCT', latitude=57.2019, longitude=-2.19778, airports=['EGPD'], weight=29)
Node 4: Node(id='Aix en Provence, FR', name='Aix en Provence', iso_country='FR', iso_region='FR-PAC', latitude=43.506294, longitude=5.366426, airports=['LFMA'], weight=2)
Node 5: Node(id='Ajaccio, FR', name='Ajaccio', iso_country='FR', iso_region='FR-COR', latitude=41.923599, longitude=8.80292, airports=['LFKJ'], weight=15)


In [9]:
import traceback
try:
    import graph_creation.adapters.input.graph_json_loader
    print('imported package graph_creation')
except Exception:
    traceback.print_exc()

imported package graph_creation
